In [1]:
import numpy as np
import pandas as pd

In [ ]:
from sklearn.impute import SimpleImputer #use for missing values filling
from sklearn.preprocessing import OneHotEncoder #use for categorical data encoding have order in data like low, medium, high
from sklearn.preprocessing import OrdinalEncoder #user for categorical nominal data encoding have no order in data like red, green, blue
from sklearn.preprocessing import StandardScaler #use for feature scaling mean=0 and variance=1 which is important for many machine learning algorithms to perform well this scales the features to have a mean of 0 and a standard deviation of 1
from sklearn.compose import ColumnTransformer #use for applying different transformations to different columns of the dataset this allows you to specify which columns should be transformed using which transformers, making it easier to preprocess the data

In [3]:
df = pd.read_csv('covid_toy.csv')

In [4]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [ ]:
df.isnull().sum() #check for missing values

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [ ]:
from sklearn.model_selection import train_test_split #model selection means splitting the dataset into training and testing sets this is important to evaluate the performance of the machine learning model on unseen data
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)#means has covid is target variable and rest are features and test size is 20% of the dataset means have  all column like 
# 1. df.drop(columns=['has_covid'])
# 👉 This removes the target column from your dataset

In [6]:
X_train

,age,gender,fever,cough,city
73,34,Male,98.0,Strong,Kolkata
38,49,Female,101.0,Mild,Delhi
1,27,Male,100.0,Mild,Delhi
97,20,Female,101.0,Mild,Bangalore
70,68,Female,101.0,Strong,Delhi
...,...,...,...,...,...
83,17,Female,104.0,Mild,Kolkata
92,82,Female,102.0,Strong,Kolkata
40,49,Female,102.0,Mild,Delhi
53,83,Male,98.0,Mild,Delhi


In [7]:
y_train

73    Yes
38    Yes
1     Yes
97     No
70     No
     ... 
83     No
92     No
40     No
53    Yes
41    Yes
Name: has_covid, Length: 80, dtype: object

## 1. Aam Zindagi

In [9]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])#default strategy is mean for numerical data and most_frequent for categorical data but here we have numerical data so it will fill the missing values with mean of the fever column in this case 
X_train_fever
# also the test data
X_test_fever = si.fit(X_test[['fever']])
                                 
X_train_fever.shape

(80, 1)

In [10]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
oe
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [15]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first',sparse_output=False)

#dense arry example
arr = np.array([[0,1],[1,0],[1,1]])
arr
ohe.fit_transform(arr)
#drop='first' means it will drop the first category of each feature to avoid multicollinearity and sparse=False means it will return a dense array instead of a sparse matrix which is easier to work with in this case
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

X_train_gender_city.shape

(80, 4)

In [19]:
arr
# ohe.fit_transform(arr)


array([[0, 1],
       [1, 0],
       [1, 1]])

In [89]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [92]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

## Mentos Zindagi

In [21]:
from sklearn.compose import ColumnTransformer

In [24]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),#cough column has two categories mild and strong so we are using ordinal encoder to encode them as 0 and 1 respectively
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [97]:
transformer.fit_transform(X_train).shape

(80, 7)

In [99]:
transformer.transform(X_test).shape

(20, 7)